# 单手牌解析分析工具

本 Notebook 用于解析单个手牌历史文件，生成 `HandHistory` 和 `State` 对象，并打印详细信息。

In [2]:
# 导入依赖
from pathlib import Path
from bayes_poker.hand_history.parse_gg_poker import (
    RushCashPokerStarsParser,
    parse_value_in_cents,
    sanitize_hand_text,
)
from pokerkit import NoLimitTexasHoldem

## 1. 输入手牌文件路径

修改下方 `hand_path` 变量，指向你要分析的 `.txt` 文件。

In [3]:
# 配置: 输入手牌文件路径
hand_path = Path("../../data/sample_hand.txt")  # <- 修改为你的手牌文件路径

# 或直接粘贴手牌文本
hand_text_raw = """
PokerStars Hand #02810746774: Hold'em No Limit ($0.01/$0.02) - 2024/08/18 01:04:56
Table 'GG_RushAndCash6198380' 6-max Seat #1 is the button
Seat 1: Stanshy101 ($2.21 in chips)
Seat 2: Lucia88 ($1.30 in chips)
Seat 3: rafer28 ($2.29 in chips)
Seat 4: JuanJoT2 ($1.41 in chips)
Seat 5: Miss Chimuelita ($2.11 in chips)
Seat 6: Marshmallowow ($2.20 in chips)
Cash Drop to Pot : total $0.20
Lucia88: posts small blind $0.01
rafer28: posts big blind $0.02
*** HOLE CARDS ***
JuanJoT2: folds
Miss Chimuelita: raises $0.03 to $0.05
Marshmallowow: calls $0.05
Stanshy101: raises $2.16 to $2.21 and is all-in
Lucia88: folds
rafer28: raises $0.08 to $2.29 and is all-in
Miss Chimuelita: folds
Marshmallowow: folds
Uncalled bet ($0.08) returned to rafer28
rafer28: shows [Ad Kd]
Stanshy101: shows [9h 9s]
*** FLOP *** [Ah Qs Jh]
rafer28: Chooses to EV Cashout
*** TURN *** [Ah Qs Jh] [2h]
*** RIVER *** [Ah Qs Jh 2h] [4d]
rafer28: Pays Cashout Risk ($0.66)
*** SHOWDOWN *** 
rafer28 collected $4.64 from pot
*** SUMMARY ***
Total pot $4.73 | Rake $0.06 | Jackpot $0.03 | Bingo $0 | Fortune $0 | Tax $0
Board [Ah Qs Jh 2h 4d]
Seat 1: Stanshy101 (button) showed [9h 9s] and lost
Seat 2: Lucia88 (small blind) folded before Flop
Seat 3: rafer28 (big blind) showed [Ad Kd] and won ($4.64), Cashout Risk ($0.66)
Seat 4: JuanJoT2 folded before Flop (didn't bet)
Seat 5: Miss Chimuelita folded before Flop
Seat 6: Marshmallowow folded before Flop


""".strip()

## 2. 加载并清洗手牌文本

In [4]:
# 从文件读取或使用直接粘贴的文本
if hand_path.exists():
    hand_text = hand_path.read_text(encoding="utf-8")
    print(f"从文件读取: {hand_path}")
else:
    hand_text = hand_text_raw
    print("使用直接粘贴的手牌文本")

# 清洗文本 (移除非标准行、处理 Cash Drop 等)
sanitized_text = sanitize_hand_text(hand_text)

print("\n--- 清洗后的手牌文本 ---")
print(sanitized_text)

使用直接粘贴的手牌文本

--- 清洗后的手牌文本 ---
PokerStars Hand #02810746774: Hold'em No Limit ($0.01/$0.02) - 2024/08/18 01:04:56
Table 'GG_RushAndCash6198380' 6-max Seat #1 is the button
Seat 1: Stanshy101 ($2.21 in chips)
Seat 2: Lucia88 ($1.30 in chips)
Seat 3: rafer28 ($2.29 in chips)
Seat 4: JuanJoT2 ($1.41 in chips)
Seat 5: Miss Chimuelita ($2.11 in chips)
Seat 6: Marshmallowow ($2.20 in chips)
Cash Drop to Pot : total $0.20
Lucia88: posts small blind $0.01
rafer28: posts big blind $0.02
*** HOLE CARDS ***
JuanJoT2: folds
Miss Chimuelita: raises $0.03 to $0.05
Marshmallowow: calls $0.05
Stanshy101: raises $2.16 to $2.21 and is all-in
Lucia88: folds
rafer28: calls $0.08
Miss Chimuelita: folds
Marshmallowow: folds
rafer28: shows [Ad Kd]
Stanshy101: shows [9h 9s]
*** FLOP *** [Ah Qs Jh]
*** TURN *** [Ah Qs Jh] [2h]
*** RIVER *** [Ah Qs Jh 2h] [4d]
*** SHOWDOWN *** 
rafer28 collected $4.64 from pot
*** SUMMARY ***
Total pot $4.73 | Rake $0.06 | Jackpot $0.03 | Bingo $0 | Fortune $0 | Tax $0
Board [Ah

## 3. 解析为 HandHistory 对象

In [5]:
parser = RushCashPokerStarsParser()

try:
    hh = parser._parse(sanitized_text, parse_value=parse_value_in_cents)
    print("✅ 解析成功!")
except Exception as e:
    print(f"❌ 解析失败: {e}")
    hh = None

✅ 解析成功!


## 4. 打印 HandHistory 详情

In [5]:
if hh:
    print("=" * 60)
    print("HandHistory 属性")
    print("=" * 60)
    print(f"玩家列表: {hh.players}")
    print(f"起始筹码: {hh.starting_stacks}")
    print(f"最终奖金: {hh.winnings}")
    print(f"抽水 (Rake): {hh.rake}")
    print(f"\n动作序列 (前20条):")
    for i, action in enumerate(hh.actions[:20]):
        print(f"  {i}: {action}")
    if len(hh.actions) > 20:
        print(f"  ... (共 {len(hh.actions)} 条动作)")

HandHistory 属性
玩家列表: ['Juandiecv', 'SlavyaniusP', 'FuLuShou poker', 'LanaLight', 'gravesJG', 'wachang']
起始筹码: [128, 200, 412, 214, 207, 200]
最终奖金: [279, 0, 0, 0, 0, 0]
抽水 (Rake): functools.partial(<function rake at 0x72e0443945e0>, percentage=0)

动作序列 (前20条):
  0: d dh p1 ????
  1: d dh p2 ????
  2: d dh p3 ????
  3: d dh p4 ????
  4: d dh p5 ????
  5: d dh p6 ????
  6: p3 cbr 4
  7: p4 f
  8: p5 cc
  9: p6 cc
  10: p1 cbr 42
  11: p2 cbr 200
  12: p3 f
  13: p5 f
  14: p6 f
  15: p1 cc
  16: p2 sm ????
  17: p1 sm ????
  18: d db Qc8s7c
  19: p1 sm ????
  ... (共 27 条动作)


## 5. 生成并重放 State 对象

In [7]:
if hh:
    print("=" * 60)
    print("重放 State 对象")
    print("=" * 60)
    
    # 创建初始 State
    state = hh.create_state()
    
    print(f"\n初始状态:")
    print(f"  盲注结构: SB={state.blinds_or_straddles}")
    print(f"  按钮位置: {state.button_index}")
    print(f"  玩家筹码: {state.stacks}")
    
    # 重放所有动作
    print(f"\n重放动作:")
    for i, action in enumerate(hh.actions):
        try:
            # 解析并执行动作
            state(action)
            # 打印关键状态变化
            if i < 10 or "d db" in str(action):  # 只打印前10条或发牌
                print(f"  [{i}] {action}")
                print(f"       底池: {state.total_pot_amount}, 筹码: {state.stacks}")
        except Exception as e:
            print(f"  [{i}] ❌ 执行失败: {action} -> {e}")
            break
    
    print(f"\n最终状态:")
    print(f"  底池: {state.total_pot_amount}")
    print(f"  筹码: {state.stacks}")
    print(f"  状态完成: {state.status.name if hasattr(state, 'status') else 'N/A'}")

重放 State 对象

初始状态:
  盲注结构: SB=(1, 2, 0, 0, 0, 0)


AttributeError: 'State' object has no attribute 'button_index'

## 6. 可视化 (可选)

你可以在这里添加更多可视化代码，例如绘制筹码变化图等。

In [9]:
# 示例: 打印公共牌
if hh:
    board_actions = [str(a) for a in hh.actions if "d db" in str(a)]
    if board_actions:
        print("公共牌发牌动作:")
        for b in board_actions:
            print(f"  {b}")
    else:
        print("未发现公共牌 (可能在 Flop 前结束)")

公共牌发牌动作:
  d db 8h8cJh
  d db Qc
